Statistical Analysis & ML Prep

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

events = spark.table("workspace.ecommerce.silver_events")
events.printSchema()


root
 |-- event_time: timestamp (nullable = true)
 |-- event_type: string (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- category_id: long (nullable = true)
 |-- category_code: string (nullable = true)
 |-- brand: string (nullable = true)
 |-- price: double (nullable = true)
 |-- user_id: integer (nullable = true)
 |-- user_session: string (nullable = true)
 |-- ingestion_ts: timestamp (nullable = true)
 |-- event_date: date (nullable = true)
 |-- price_tier: string (nullable = true)



In [0]:
events.describe("price").show()


+-------+------------------+
|summary|             price|
+-------+------------------+
|  count|          42341904|
|   mean|290.78126383905595|
| stddev| 358.3993774731591|
|    min|              0.77|
|    max|           2574.07|
+-------+------------------+



In [0]:
events = events.withColumn(
    "is_weekend",
    F.dayofweek("event_date").isin([1,7])  # Sunday=1, Saturday=7
)


In [0]:
events.groupBy("is_weekend", "event_type").count().orderBy("is_weekend").show()


+----------+----------+--------+
|is_weekend|event_type|   count|
+----------+----------+--------+
|     false|      view|29711315|
|     false|      cart|  642243|
|     false|  purchase|  546365|
|      true|      cart|  253320|
|      true|      view|10992259|
|      true|  purchase|  196402|
+----------+----------+--------+



In [0]:
conversion = (
    events.groupBy("is_weekend")
    .agg(
        F.count(F.when(F.col("event_type")=="view", True)).alias("views"),
        F.count(F.when(F.col("event_type")=="purchase", True)).alias("purchases")
    )
    .withColumn(
        "conversion_rate",
        F.round(F.col("purchases")*100/F.col("views"), 2)
    )
)

conversion.show()


+----------+--------+---------+---------------+
|is_weekend|   views|purchases|conversion_rate|
+----------+--------+---------+---------------+
|      true|10992259|   196402|           1.79|
|     false|29711315|   546365|           1.84|
+----------+--------+---------+---------------+



In [0]:
events = events.withColumn(
    "is_purchase",
    F.when(F.col("event_type")=="purchase", 1).otherwise(0)
)


In [0]:
events.stat.corr("price", "is_purchase")


0.0070015989652335315

In [0]:
features = (
    events
    .withColumn("hour", F.hour("event_time"))
    .withColumn("day_of_week", F.dayofweek("event_date"))
    .withColumn("price_log", F.log(F.col("price") + 1))
)


In [0]:
window = Window.partitionBy("user_id").orderBy("event_time")

features = features.withColumn(
    "time_since_first_event",
    F.unix_timestamp("event_time") -
    F.unix_timestamp(F.first("event_time").over(window))
)